In [ ]:
import tensorflow as tf
import sys
import os
from typing import List, Tuple, Dict, Union, Optional, Any


'''
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
    sys.path.insert(1, parent_dir)
'''
sys.path.append('../')

from ConditionalBijectorWrapper import Cond_RealNVP
from RealNVP import *

In [ ]:
ndims = 5
rem_dims = 1
ncond = 4
n_hidden=[50,50]

In [ ]:
#input_structure = "separated"
cond_realnvp = Cond_RealNVP(ndims, rem_dims, conditional_event_shape=(1,1,1,1,), input_structure = "SuperCalo", conditional_input_layers = 'all_layers')


In [ ]:

x = tf.constant([[3.0, 2.3, 2.0, 4.5, 1.2]], dtype=tf.float32)
cond = tf.constant([[1.0, 1.0, 1.0, 1.0,]], dtype=tf.float32)
x_cond = tf.concat([x, cond], axis=-1)
out1 = cond_realnvp._forward(x_cond)
out2 = cond_realnvp._inverse(out1)
print(out1.numpy())

# Conditional permute

In [ ]:
import tensorflow as tf
import tensorflow_probability as tfp

# Alias
tfb = tfp.bijectors

class CondPermute(tfb.Bijector):
    """
    Conditional Permute Bijector: applies a fixed permutation to the first `data_dims` of the input,
    leaving the last `cond_dims` dimensions unchanged.
    """
    def __init__(self,
                 perm,
                 cond_dims,
                 validate_args=False,
                 name="cond_permute"):  # forward_min_event_ndims defaults to 1
        """
        Args:
            perm: Integer `Tensor` of shape [data_dims], a permutation of [0..data_dims-1].
            cond_dims: Number of conditional dimensions at the end of the event.
        """
        self.perm = tf.convert_to_tensor(perm, dtype=tf.int32)
        self.cond_dims = cond_dims
        data_dims = tf.shape(self.perm)[0]
        # Build inverse permutation map
        inv = tf.scatter_nd(
            indices=tf.expand_dims(self.perm, -1),
            updates=tf.range(data_dims, dtype=tf.int32),
            shape=[data_dims]
        )
        super().__init__(
            forward_min_event_ndims=1,
            is_constant_jacobian=True,
            validate_args=validate_args,
            name=name
        )
        self._inverse_perm = inv
        print("__________________________________________permuting")

    def _forward(self, x):
        # x: [..., data_dims + cond_dims]
        data, cond = x[..., :-self.cond_dims], x[..., -self.cond_dims:]
        permuted = tf.gather(data, self.perm, axis=-1)
        return tf.concat([permuted, cond], axis=-1)

    def _inverse(self, y):
        data, cond = y[..., :-self.cond_dims], y[..., -self.cond_dims:]
        unpermuted = tf.gather(data, self._inverse_perm, axis=-1)
        return tf.concat([unpermuted, cond], axis=-1)

    def _forward_log_det_jacobian(self, x):
        # Permutation has zero log-det
        return tf.zeros(tf.shape(x)[:-1], dtype=x.dtype)

    def _inverse_log_det_jacobian(self, y):
        return tf.zeros(tf.shape(y)[:-1], dtype=y.dtype)


# Chain bijectors

In [ ]:

def Shufflefirst(mask,ndims,rem_dims):
    k=0
    permutation=[]
    zeros=[]
    ones=[]
    
    for  elem in mask:
        if elem==0:
            zeros.append(k)
        if elem==1:
            ones.append(k)
        k=k+1
        
    print(zeros)
    print(ones)
    print(zeros+ones)
    permutation=tf.cast(zeros+ones,dtype=tf.int32)
    return permutation

def Log2D(ndims):
    nlog2d=int(np.log2(ndims))
    if 2**nlog2d==ndims:
        n_bijectors=nlog2d
    else:
        n_bijectors=1+nlog2d

    return n_bijectors

def DecimalToBinary(ndims,n_bijectors):

    binaries_list=[]
    for dec in range(ndims):
        biny= bin(dec).replace("0b", "").zfill(n_bijectors)
        binaries_list.append(biny)
        
    return binaries_list

def RandomShuffle(ndims):

    arr = np.arange(ndims)
    np.random.shuffle(arr)
    random_shuffle=tf.cast(arr, tf.int32)
    return random_shuffle

def ReverseShuffle(ndims):

    arr = np.arange(ndims)
    arr=np.flip(arr)
    reverse_shuffle=tf.cast(arr, tf.int32)
    return reverse_shuffle

def RealNVPN(ndims,rem_dims,num_bijectors,hidden_layers,activation, cond_dims=0, input_structure = None, use_bias=True,
    kernel_initializer='glorot_uniform',
    bias_initializer='zeros', kernel_regularizer=None,
    bias_regularizer=None, activity_regularizer=None, kernel_constraint=None,
    bias_constraint=None,perm_style='bi-partition',shuffle='Noshuffle'):

    
    
    if perm_style=='bi-partition':
   
        permutation=tf.cast(np.concatenate((np.arange(int(ndims/2),ndims),np.arange(0,int(ndims/2)))), tf.int32)
    if perm_style=='reverse':
        
        permutation=ReverseShuffle(ndims)
    
    if shuffle=='Noshuffle':
        bijectors=[]

        for i in range(num_bijectors):
            #bijectors.append(tfb.BatchNormalization())
            bijectors.append(Cond_RealNVP(ndims,rem_dims,cond_dims, hidden_layers,activation,use_bias,
            kernel_initializer,
            bias_initializer, kernel_regularizer,
            bias_regularizer, activity_regularizer, kernel_constraint,
            bias_constraint,input_structure=input_structure))
            bijectors.append(CondPermute(permutation, cond_dims))
        bijector = tfb.Chain(bijectors=list(reversed(bijectors[:-1])), name='chain_of_real_nvp')

        
    elif shuffle=='RandomShuffle':
        bijectors=[]


        for i in range(num_bijectors):
            #bijectors.append(tfb.BatchNormalization())
            bijectors.append(Cond_RealNVP(ndims,rem_dims,cond_dims, hidden_layers,activation,use_bias,
            kernel_initializer,
            bias_initializer, kernel_regularizer,
            bias_regularizer, activity_regularizer, kernel_constraint,
            bias_constraint,input_structure=input_structure))
            ###permute transformed and transforming dims
            bijectors.append(CondPermute(permutation, cond_dims))
            
            bijectors.append(Cond_RealNVP(ndims,rem_dims, cond_dims,hidden_layers,activation,use_bias,
            kernel_initializer,
            bias_initializer, kernel_regularizer,
            bias_regularizer, activity_regularizer, kernel_constraint,
            bias_constraint,input_structure=input_structure))
            ##shuffle
            bijectors.append(CondPermute(RandomShuffle(ndims)))
            

        bijector = tfb.Chain(bijectors=list(reversed(bijectors[:-1])), name='chain_of_real_nvp')

        
    elif 'BinaryWiseShuffle':
        print(perm_style)
        print(shuffle)
        n_bijectors=Log2D(ndims)
        binaries_list=DecimalToBinary(ndims,n_bijectors)
        print(binaries_list)
        bijectors=[]
        for bij in range(n_bijectors):
            mask=ShuffleMask(binaries_list,bij)
            rem_dims=GetRemDims(ndims,mask)
            
            
            BinShufflepermutation=Shufflefirst(mask,ndims,rem_dims)
            bijectors.append(tfp.bijectors.Permute(BinShufflepermutation))
            
            
            bijectors.append(Cond_RealNVP(ndims,rem_dims, cond_dims,hidden_layers,activation,use_bias,
            kernel_initializer,
            bias_initializer, kernel_regularizer,
            bias_regularizer, activity_regularizer, kernel_constraint,
            bias_constraint,input_structure=input_structure))
            
            bi_partiion_permutation=ShuffleSecond(ndims,rem_dims)
            bijectors.append(tfp.bijectors.Permute(bi_partiion_permutation))
            
            bijectors.append(Cond_RealNVP(ndims,rem_dims, cond_dims,hidden_layers,activation,use_bias,
            kernel_initializer,
            bias_initializer, kernel_regularizer,
            bias_regularizer, activity_regularizer, kernel_constraint,
            bias_constraint,input_structure=input_structure))
        print(bijectors)
        bijector = tfb.Chain(bijectors=list(reversed(bijectors[:-1])), name='chain_of_real_nvp')
    
    return bijector


# Trainer helper functions


In [ ]:
import Trainer


def get_compiler_kwargs(lr: float,
                        ignore_nans: bool,
                        nan_threshold: float
                       ):
    #compiler_kwargs = {'optimizer': {'class_name': 'Adam', # this gives the new Adam optimizer
    compiler_kwargs = {'optimizer': {'class_name': 'Custom>Adam', # this gives the new Adam optimizer
                                     'config': {'learning_rate': lr,
                                                'beta_1': 0.9,
                                                'beta_2': 0.999,
                                                'epsilon': 1e-07,
                                                'amsgrad': True}},
                       'metrics': [{'class_name': 'MinusLogProbMetric',
                                    'config': {'ignore_nans': ignore_nans,
                                               'debug_print_mode': False}}],
                       'loss': {'class_name': 'MinusLogProbLoss',
                                'config': {'name': "MLP",
                                           'ignore_nans': ignore_nans,
                                           'nan_threshold': nan_threshold,
                                           'debug_print_mode': False}}}
    return compiler_kwargs

def get_callbacks_kwargs(#checkpoint_path: str,
                         es_min_delta: float,
                         es_patience: int,
                         lr_reduce_factor: float,
                         lr_min_delta: float,
                         lr_patience: int,
                         min_lr: float
                         ):
    callbacks_kwargs = [{'class_name': 'PrintEpochInfo',
                         'config': {}},
                        #{'class_name': 'HandleNaNCallback',
                        # 'config': {'checkpoint_path': checkpoint_path,
                        #            'lr_reduction_factor': lr_reduce_factor_on_nan,
                        #            'random_seed_var': np.random.randint(1000000)}},
                        #{'class_name': 'TerminateOnNaNFractionCallback',
                        # 'config': {'threshold': 0.1,
                        #            'validation_data': X_data_val}},
                        #{'class_name': 'ModelCheckpoint',
                        # 'config': {#'filepath': checkpoint_path,
                        #            'monitor': 'val_loss',
                        #            'save_best_only': True,
                        #            'save_weights_only': True,
                        #            'verbose': 1,
                        #            'mode': 'auto',
                        #            'save_freq': 'epoch'}},
                        {'class_name': 'EarlyStopping',
                         'config': {'monitor': 'val_loss',
                                    'min_delta': es_min_delta,
                                    'patience': es_patience,
                                    'verbose': 1,
                                    'mode': 'auto',
                                    'baseline': None,
                                    'restore_best_weights': True}},
                        {'class_name': 'ReduceLROnPlateau',
                         'config': {'monitor': 'val_loss',
                                    'factor': lr_reduce_factor,
                                    'min_delta': lr_min_delta,
                                    'patience': lr_patience,
                                    'min_lr': min_lr}},
                        {'class_name': 'TerminateOnNaN', 'config': {}}
                        ]
    return callbacks_kwargs

def get_fit_kwargs(batch_size: int,
                   epochs_input: int,
                   validation_data: Tuple[Union[np.ndarray,tf.Tensor],Union[np.ndarray,tf.Tensor]],
                   shuffle: bool,
                   verbose: int
                  ) -> Dict[str,Any]:
    fit_kwargs = {'batch_size': batch_size,
                  'epochs': epochs_input,
                  'validation_data': validation_data,
                  'shuffle': shuffle,
                  'verbose': verbose}
    return fit_kwargs

In [ ]:
def gen_data2(means, stddevs, nsamples, ndims, ncomp, seed=0):

  tf.random.set_seed(seed)
  means = tf.cast(means, tf.float32)
  stddevs = tf.cast(stddevs, tf.float32)
  #questo fa che ogni riga è un sample. Per ogni sample (colonne) genera un intero da 0 a ncomp-1 per ogni dimensione
  labels = tf.random.uniform(shape=(nsamples, ndims), minval=0, maxval=ncomp, dtype=tf.int32)

  # Build gather indices: shape (nsamples, ndim, 2) where each index is [dim, label]
  dim_indices = tf.range(ndims)[tf.newaxis, :]                 # shape (1, ndim)
  dim_indices = tf.tile(dim_indices, [nsamples, 1])            # shape (nsamples, ndim)
  gather_indices = tf.stack([dim_indices, labels], axis=-1)    # shape (nsamples, ndim, 2)

  # Gather mean and std for each (sample, dimension) from (ndim, ncomp)
  chosen_means = tf.gather_nd(means, gather_indices)           # shape (nsamples, ndim)
  chosen_stds = tf.gather_nd(stddevs, gather_indices)          # shape (nsamples, ndim)

  # Sample from standard normal and scale/shift
  samples = tf.random.normal(shape=(nsamples, ndims)) * chosen_stds + chosen_means  # shape (nsamples, ndim)

  # Combine samples and labels: shape (nsamples, ndim, 2)
  data_tensor = tf.concat([samples, tf.cast(labels, samples.dtype)], axis=-1)
  label_tensor = labels

  Y_data: tf.Tensor = tf.zeros((data_tensor.shape[0], 0), dtype=data_tensor.dtype)

  return data_tensor, Y_data

### Bijector initializzation

In [ ]:
ndims = 2
rem_dims = 1
ncond = 2
n_hidden=[100,100,100,100]
num_bijectors=4
activation='relu'
#bijector = RealNVPN(ndims,rem_dims,num_bijectors,n_hidden,activation, cond_dims=ncond, input_structure=None)
#for i, b in enumerate(bijector.bijectors):
#    b._name = f"{b.name}_{i}"


In [ ]:
from Bijectors import *

bijector_name = 'Cond_RealNVPN'
spline_knots = 2
range_min = 10
eps_regulariser = 1e-6
regulariser = 'l2'
bijector = ChooseCondBijector(bijector_name, ndims, spline_knots, num_bijectors, range_min, n_hidden, activation, regulariser, eps_regulariser, conditional_event_shape=(1,1,), input_structure = None)

In [ ]:
'''
total_dims = ndims + ncond
y = tf.random.normal([4, total_dims])   # e.g. batch=4, total_dims=4

z = y
tf.print("Start y:", tf.shape(z))
for b in bijector.bijectors:
  try:
    z = b.inverse(z)
    tf.print("After", b.name, "→", tf.shape(z))
  except Exception as e:
    tf.print("❌ Failed at", b.name, "with error:", e.message)
    break
'''

### Data generation

In [ ]:
nsamples = 10000
nsamples_val = 1000
ncomp = 2
means = tf.constant([[-2.0, 2.0],     # dim 0: two component means
                     [-3.0, 3.0]])    # dim 1: two component means
stddevs = tf.constant([[0.5, 0.5],
                       [1.0, 1.0]])

x_data, y_data = gen_data2(means, stddevs, nsamples, ndims, ncomp)
x_data_val, y_data_val = gen_data2(means, stddevs, nsamples_val, ndims,ncomp)
x_data.numpy().shape

### Trainer initializzation


In [ ]:
base_dist = tfd.MultivariateNormalDiag(loc=tf.zeros(ndims), scale_diag=tf.ones(ndims))
#base_dist  = tfd.JointDistributionSequential([base_x, tfd.Deterministic(cond_vector)])

In [ ]:
### Compiler hyperparameters ###
lr: float = 0.001
ignore_nans: bool = True
nan_threshold: float = 0.01

### Initialize callbacks hyperparameters ###
es_min_delta: float = .0001
es_patience: int = 100
lr_reduce_factor: float = .5
lr_min_delta: float = .0001
lr_patience: int = 50
min_lr: float = 1e-6

### Initialzie training hyperparameters ###
batch_size: int = 128
epochs_input: int = 10
shuffle: bool = True
verbose_trainer: int = 2

### Debugging parameter
debug_print_mode: bool = False


In [ ]:
NFObject: Trainer.Trainer = Trainer.Trainer(base_distribution = base_dist,
                                            flow = bijector, 
                                            x_data_train = x_data,
                                            y_data_train = y_data,
                                            compiler_kwargs = get_compiler_kwargs(lr = lr,
                                                                                  ignore_nans = True,
                                                                                  nan_threshold = nan_threshold),
                                            callbacks_kwargs = get_callbacks_kwargs(#checkpoint_path = checkpoint_path,
                                                                                    es_min_delta = es_min_delta,
                                                                                    es_patience = es_patience,
                                                                                    lr_reduce_factor = lr_reduce_factor,
                                                                                    lr_min_delta = lr_min_delta,
                                                                                    lr_patience = lr_patience,
                                                                                    min_lr = min_lr),
                                            fit_kwargs = get_fit_kwargs(batch_size = batch_size,
                                                                        epochs_input = epochs_input,
                                                                        validation_data = (x_data_val, y_data_val),
                                                                        shuffle = shuffle,
                                                                        verbose = verbose_trainer),
                                            debug_print_mode = debug_print_mode)
trainable_params: int = NFObject.trainable_params
non_trainable_params: int = NFObject.non_trainable_params

In [ ]:
NFObject.train()

In [ ]:
nf_dist  = NFObject.nf_dist

In [ ]:
import tensorflow as tf

def sample_conditional(nf_dist, ncond, nsamples: int, conditionals: tf.Tensor):
    """
    Draw nsamples from p(x | conditionals) via
      x = flow.forward(z, conditionals)
    where z ~ base_dist.
    
    Args:
      nf_dist:           a tfp.distributions.TransformedDistribution
                         wrapping (base_dist, bijector=flow)
      nsamples:          int, number of draws
      conditionals:      Tensor of shape (ncond,), dtype float32
                         the single conditional vector you want to condition on.
    
    Returns:
      samples: Tensor of shape (nsamples, ndims),
               the generated continuous data samples.
    """
    flow      = nf_dist.bijector
    base_dist = nf_dist.distribution

    # 1) sample latent z ~ base_dist; shape (nsamples, ndims)
    z = base_dist.sample(nsamples)  # tf.Tensor, shape [nsamples, ndims]

    # 2) make a [nsamples, ncond] matrix of your condition
    conditionals = tf.convert_to_tensor(conditionals, dtype=z.dtype)
    conds = tf.broadcast_to(conditionals[None, :], [nsamples, ncond])

    # 3) concatenate [z, conds] → shape (nsamples, ndims+ncond)
    z_and_c = tf.concat([z, conds], axis=-1)

    # 4) push forward through the flow
    x_and_c = flow.forward(z_and_c)  # shape [nsamples, ndims+ncond]

    # 5) extract just the data dims
    return x_and_c[:, :base_dist.event_shape[0]]


In [ ]:
ns = 10000
conditionals = tf.constant([1.0, 0.0], dtype=tf.float32)  # your 2‐dim condition
samples = sample_conditional(nf_dist, ncond, ns, conditionals)
print(samples.shape)  # (1000, ndims)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

fig, axs = plt.subplots(1, 2, figsize=(12, 5))

for dim in range(2):
    selected_label = int(conditionals[dim])  # 0 or 1

    for comp in range(2):
        mean = means[dim, comp]
        std = stddevs[dim, comp]

        x_vals = np.linspace(mean - 4 * std, mean + 4 * std, 200)
        if comp == selected_label:
            axs[dim].plot(x_vals, norm.pdf(x_vals, mean, std), 'r-', lw=2.5, label=f'Selected N({mean}, {std}²)')
        else:
            axs[dim].plot(x_vals, norm.pdf(x_vals, mean, std), 'k--', lw=1.0, label=f'Other N({mean}, {std}²)')

    axs[dim].hist(samples[:, dim], bins=28, density=True, alpha=0.6, label='Sampled Data')
    axs[dim].set_title(f"Dimension {dim}")
    axs[dim].legend()
    axs[dim].set_xlabel("Value")
    axs[dim].set_ylabel("Density")

plt.tight_layout()
plt.show()